# Lecture 3: Vector Calculus in Computer Graphics
**CMU 15-462/662 — Interactive Companion Notebook**

This notebook accompanies the `cg-03-lecture-quiz.md` flashcard deck.  
Each section corresponds to quiz questions and provides runnable code + matplotlib visualizations.

**Topics:**  Euclidean norms · Inner & cross products · Derivatives · Gradient · Divergence · Curl · Laplacian · Hessian

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor':   '#1e293b',
    'axes.edgecolor':   '#475569',
    'axes.labelcolor':  '#94a3b8',
    'xtick.color':      '#64748b',
    'ytick.color':      '#64748b',
    'text.color':       '#f1f5f9',
    
    'grid.color':       '#334155',
    'grid.alpha':       0.4,
    'axes.grid':        True,
})
BLUE   = '#38bdf8'
GREEN  = '#10b981'
ORANGE = '#f59e0b'
RED    = '#ef4444'
PURPLE = '#818cf8'

: 

---
## 1  Euclidean Norm  [Q1–Q5]

**Key fact:** The Euclidean norm `‖u‖ = √(Σuᵢ²)` is valid *only* when the coordinates come from an **orthonormal basis**.  
In a skewed basis the formula gives the wrong answer.

In [ ]:
# ── Q4/Q5  Orthonormal vs skewed basis ──────────────────────────────────────

# Standard orthonormal basis
e1_on = np.array([1., 0.])
e2_on = np.array([0., 1.])

# Skewed basis: e1=(1,0), e2=(0.6, 0.8)  (not perpendicular)
e1_sk = np.array([1.,  0.])
e2_sk = np.array([0.6, 0.8])

coords = np.array([1., 1.])   # same coordinates in both bases

vec_on = coords[0]*e1_on + coords[1]*e2_on   # geometric vector in ortho basis
vec_sk = coords[0]*e1_sk + coords[1]*e2_sk   # geometric vector in skewed basis

print(f'Coord formula (both):  {np.linalg.norm(coords):.4f}')
print(f'True norm (ortho):     {np.linalg.norm(vec_on):.4f}  ← matches')
print(f'True norm (skewed):    {np.linalg.norm(vec_sk):.4f}  ← coord formula is WRONG')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle('Euclidean Norm: Orthonormal vs Skewed Basis', color='#f1f5f9', fontsize=13)

for ax, e1, e2, vec, title, basis_type in [
    (ax1, e1_on, e2_on, vec_on, 'Orthonormal', 'ortho'),
    (ax2, e1_sk, e2_sk, vec_sk, 'Skewed', 'skewed'),
]:
    ax.set_title(title, color='#f1f5f9')
    ax.set_xlim(-0.2, 2.2); ax.set_ylim(-0.2, 2.2)
    ax.set_aspect('equal')
    ax.axhline(0, color='#334155'); ax.axvline(0, color='#334155')
    ax.annotate('', xy=e1, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=BLUE,  lw=2))
    ax.annotate('', xy=e2, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=GREEN, lw=2))
    ax.annotate('', xy=vec, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=ORANGE, lw=2.5))
    ax.text(e1[0]+0.05, e1[1]+0.05, 'e₁', color=BLUE, fontsize=12)
    ax.text(e2[0]+0.05, e2[1]+0.05, 'e₂', color=GREEN, fontsize=12)
    ax.text(vec[0]+0.05, vec[1]+0.05, f'v  ‖v‖={np.linalg.norm(vec):.2f}', color=ORANGE, fontsize=11)

plt.tight_layout()
plt.show()

: 

---
## 2  Inner Product & Cross Product  [Q6–Q15]

**Inner product:** `<u,v> = ‖u‖‖v‖cos θ`  
**Cross product:** magnitude = area of parallelogram; direction = right-hand rule

In [ ]:
# ── Q8/Q9  Cross product magnitude = parallelogram area ──────────────────────

u = np.array([2., 0., 0.])
v = np.array([1., 2., 0.])

cross = np.cross(u, v)
area_formula = np.linalg.norm(cross)
area_base_height = np.linalg.norm(u) * np.linalg.norm(v) * np.sin(np.arccos(
    np.dot(u,v) / (np.linalg.norm(u)*np.linalg.norm(v))
))

print(f'u = {u},  v = {v}')
print(f'u × v = {cross}')
print(f'‖u × v‖ = {area_formula:.4f}  (parallelogram area)')
print(f'base × height = {area_base_height:.4f}  ✓')

fig, ax = plt.subplots(figsize=(7, 5))
ax.set_title('Cross Product — Parallelogram Area', color='#f1f5f9')
origin = np.zeros(2)
ax.annotate('', xy=u[:2], xytext=origin, arrowprops=dict(arrowstyle='->', color=BLUE, lw=2))
ax.annotate('', xy=v[:2], xytext=origin, arrowprops=dict(arrowstyle='->', color=GREEN, lw=2))
from matplotlib.patches import Polygon
para = np.array([[0,0], u[:2], (u+v)[:2], v[:2]])
ax.add_patch(Polygon(para, closed=True, alpha=0.25, color=ORANGE))
ax.text(u[0]/2, -0.15, 'u', color=BLUE, fontsize=13)
ax.text(v[0]-0.3, v[1]/2, 'v', color=GREEN, fontsize=13)
ax.text(1.3, 0.9, f'Area = {area_formula:.2f}', color=ORANGE, fontsize=12)
ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, 2.5); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

In [ ]:
# ── Q13  Hat (skew-symmetric) matrix ─────────────────────────────────────────

def hat(u):
    """Skew-symmetric matrix such that hat(u) @ v == np.cross(u, v)"""
    return np.array([[   0,  -u[2],  u[1]],
                     [ u[2],    0,  -u[0]],
                     [-u[1],  u[0],    0]])

u = np.array([1., 2., 3.])
v = np.array([4., 5., 6.])

print('Hat matrix [u]× =')
print(hat(u))
print()
print('hat(u) @ v      =', hat(u) @ v)     # should match
print('np.cross(u, v)  =', np.cross(u, v))  # same result
print()
# Anti-commutativity: u×v = -v×u
print('u×v =', np.cross(u,v))
print('v×u =', np.cross(v,u), '  (negated ✓)')

---
## 3  Derivatives  [Q14–Q19]

The derivative is the **best linear approximation** at a point.  
Directional derivative: `D_u f(x₀) = lim_{ε→0} [f(x₀+εu) − f(x₀)] / ε`

In [ ]:
# ── Q14  Derivative as best linear approximation ─────────────────────────────

def f(x):  return np.sin(x)
def fp(x): return np.cos(x)   # analytical derivative

x0 = 1.0
x  = np.linspace(0.0, 2.2, 400)
linear_approx = f(x0) + fp(x0) * (x - x0)   # tangent line

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title('Derivative = Best Linear Approximation', color='#f1f5f9')
ax.plot(x, f(x), color=BLUE, lw=2.5, label='f(x) = sin(x)')
ax.plot(x, linear_approx, color=ORANGE, lw=2, ls='--', label=f"f(x₀) + f'(x₀)(x−x₀)")
ax.axvline(x0, color='#475569', ls=':', lw=1)
ax.scatter([x0], [f(x0)], color=RED, zorder=5, s=80)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(framealpha=0.2)
# Zoom inset to show how close they are
ax.set_xlim(0, 2.2); ax.set_ylim(-0.3, 1.4)
ax.text(1.05, f(x0)+0.05, 'x₀', color=RED)
plt.tight_layout(); plt.show()

# Numerical vs analytical
eps = 1e-5
numerical = (f(x0 + eps) - f(x0)) / eps
print(f"Analytical f'({x0}) = {fp(x0):.6f}")
print(f"Numerical  f'({x0}) = {numerical:.6f}")

In [ ]:
# ── Q16  Directional derivative ───────────────────────────────────────────────
# D_u f(x) = ∇f(x) · u   (when u is a unit vector)

def f2(x, y): return np.sin(x) * np.exp(-0.3 * y**2)

x0, y0 = 1.0, 0.5

# Finite-difference directional derivative in direction u
def dir_deriv(x0, y0, ux, uy, eps=1e-6):
    return (f2(x0+eps*ux, y0+eps*uy) - f2(x0, y0)) / eps

angles = np.linspace(0, 2*np.pi, 360)
dd = [dir_deriv(x0, y0, np.cos(a), np.sin(a)) for a in angles]

fig, ax = plt.subplots(figsize=(7, 5), subplot_kw=dict(projection='polar'))
ax.set_facecolor('#1e293b')
ax.plot(angles, dd, color=BLUE, lw=2)
ax.fill(angles, np.clip(dd, 0, None), alpha=0.2, color=BLUE)
ax.set_title('Directional derivative D_u f(x₀)\nas a function of direction', 
             color='#f1f5f9', pad=20)
ax.tick_params(colors='#64748b')
plt.tight_layout(); plt.show()
print('Max directional derivative is in gradient direction.')
print(f'Max value ≈ {max(dd):.4f}  (= ‖∇f‖ at that point)')

---
## 4  Gradient  [Q19–Q32]

**Definition:** ∇f(x) is the unique vector s.t. `<∇f, u> = D_u f` for all u.  
- Points in the direction of **steepest ascent**
- Perpendicular to **level sets** (iso-contours)

In [ ]:
# ── Q20/Q21  Gradient field and level sets ────────────────────────────────────

x, y = np.meshgrid(np.linspace(-2, 2, 80), np.linspace(-2, 2, 80))
z = np.sin(x) * np.cos(y)       # f(x, y)

# Gradient via finite differences
dz_dx, dz_dy = np.gradient(z, x[0], y[:,0])

# Subsample for quiver
step = 6
xs, ys = x[::step, ::step], y[::step, ::step]
gx, gy = dz_dx[::step, ::step], dz_dy[::step, ::step]

fig, ax = plt.subplots(figsize=(8, 7))
ax.set_title('f(x,y) = sin(x)cos(y) — Gradient Field (∇f) over Level Sets', color='#f1f5f9')
cs = ax.contourf(x, y, z, levels=20, cmap='cool', alpha=0.6)
ax.contour(x, y, z, levels=10, colors='#475569', linewidths=0.8, alpha=0.7)
qv = ax.quiver(xs, ys, gx, gy, color=ORANGE, alpha=0.85, scale=15, width=0.003)
plt.colorbar(cs, ax=ax, label='f value')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
plt.tight_layout(); plt.show()
print('Notice: gradient arrows are perpendicular to the level-set contour lines.')

In [ ]:
# ── Q26  Jacobian (matrix derivative for vector-valued f) ─────────────────────
# For f: R² → R²,  J[i,j] = ∂fᵢ/∂xⱼ

import sympy as sp

x_s, y_s = sp.symbols('x y')
f1 = x_s**2 * sp.sin(y_s)
f2_sym = sp.exp(x_s) * y_s

J = sp.Matrix([[sp.diff(f1, x_s), sp.diff(f1, y_s)],
               [sp.diff(f2_sym, x_s), sp.diff(f2_sym, y_s)]])

print('f(x,y) = [x²sin(y),  e^x · y]')
print('Jacobian J =')
sp.pprint(J)

# Evaluate at a point
pt = {x_s: 1, y_s: np.pi/4}
J_num = np.array([[float(J[i,j].subs(pt)) for j in range(2)] for i in range(2)])
print(f'\nJ at (1, π/4) =')
print(np.round(J_num, 4))

---
## 5  Divergence  [Q33–Q45]

**div F = ∇·F = ∂Fₓ/∂x + ∂Fy/∂y + ∂Fz/∂z**

Intuition: net outward flow per unit volume.  Positive = source, negative = sink.

In [ ]:
# ── Q34/Q35  Divergence: source vs sink vs zero ───────────────────────────────

x, y = np.meshgrid(np.linspace(-2, 2, 20), np.linspace(-2, 2, 20))

fields = [
    ('Source: F = (x, y)',        x,     y,    '+'  ),
    ('Sink:   F = (−x, −y)',     -x,    -y,    '−'  ),
    ('Zero:   F = (−y, x)',      -y,     x,    '0'  ),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Divergence: Sources, Sinks, and Divergence-Free Fields', color='#f1f5f9', fontsize=13)

for ax, (title, u, v, div_sign) in zip(axes, fields):
    mag = np.sqrt(u**2 + v**2) + 1e-6
    ax.quiver(x, y, u/mag, v/mag, mag, cmap='plasma', alpha=0.9)
    ax.set_title(f'{title}\ndiv = {div_sign}', color='#f1f5f9', fontsize=10)
    ax.set_aspect('equal')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)

plt.tight_layout(); plt.show()

In [ ]:
# ── Divergence theorem (Gauss): ∯_∂Ω F·n dA = ∫∫∫_Ω div F dV ────────────────
# For F=(x,y):  div F = 2.  Integral over disk r<R → area * 2 = πR²*2
# Flux through boundary circle = ∮ F·n ds = ∮ r dr dθ → same answer

R = 1.0
volume_integral = 2 * np.pi * R**2    # div=2 × area of disk

# Numerically compute boundary flux
theta = np.linspace(0, 2*np.pi, 10000)
x_b = R * np.cos(theta)
y_b = R * np.sin(theta)
nx  = np.cos(theta)   # outward normal on circle
ny  = np.sin(theta)
Fx  = x_b
Fy  = y_b
flux = np.trapz(Fx*nx + Fy*ny, theta) * R   # ∮ F·n ds

print(f'Volume integral (div theorem):  {volume_integral:.4f}')
print(f'Boundary flux (numerical):      {flux:.4f}')
print(f'Match: {np.isclose(volume_integral, flux, rtol=1e-3)}')

---
## 6  Curl  [Q46–Q55]

**curl F = ∇×F**

Measures local rotation in a vector field.  
- In 2D, curl F = ∂Fy/∂x − ∂Fx/∂y  (scalar)  
- Conservative field (F = ∇f): **curl = 0**

In [ ]:
# ── Q47/Q48  Rotational vs irrotational fields ────────────────────────────────

x, y = np.meshgrid(np.linspace(-2, 2, 20), np.linspace(-2, 2, 20))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Curl: Rotational vs Irrotational Fields', color='#f1f5f9', fontsize=13)

# Rotational field: F = (−y, x)  →  curl = ∂x/∂x − ∂(−y)/∂y = 1+1 = 2
u1, v1 = -y, x
ax1.quiver(x, y, u1, v1, np.sqrt(u1**2+v1**2), cmap='plasma')
ax1.set_title('F = (−y, x)\ncurl = 2  (rotational)', color='#f1f5f9')
ax1.set_aspect('equal'); ax1.set_xlim(-2.5,2.5); ax1.set_ylim(-2.5,2.5)

# Irrotational (gradient) field: F = ∇(x²+y²) = (2x, 2y)  →  curl = 0
u2, v2 = 2*x, 2*y
ax2.quiver(x, y, u2, v2, np.sqrt(u2**2+v2**2), cmap='plasma')
ax2.set_title('F = (2x, 2y) = ∇(x²+y²)\ncurl = 0  (irrotational)', color='#f1f5f9')
ax2.set_aspect('equal'); ax2.set_xlim(-2.5,2.5); ax2.set_ylim(-2.5,2.5)

plt.tight_layout(); plt.show()

# Numerical curl (2D) ∂Fy/∂x − ∂Fx/∂y
dx = x[0,1] - x[0,0]
dFy_dx = np.gradient(x*2, dx, axis=1)   # ∂Fy/∂x for irrotational field
dFx_dy = np.gradient(y*2, dx, axis=0)   # ∂Fx/∂y
curl_irrot = dFy_dx - dFx_dy
print(f'Max |curl| of irrotational field: {np.max(np.abs(curl_irrot)):.2e}  (≈ 0 ✓)')

---
## 7  Laplacian  [Q56–Q65]

**Δf = ∇·∇f = Σ ∂²f/∂xᵢ²**

Key interpretation: Δf(x) ≈ average of f in a small ball around x  **minus** f(x).  
- Δf > 0 : x is a **local minimum** (neighbours are higher on average)  
- Δf < 0 : x is a **local maximum**  
- Δf = 0 : **harmonic** function (mean-value property)  

The heat equation `∂u/∂t = Δu` smooths the function over time.

In [ ]:
# ── Q58  Laplacian as mean-value deviation ────────────────────────────────────

x, y = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
r2 = x**2 + y**2
f  = np.exp(-r2)   # Gaussian: has a clear maximum at origin

# Numerical Laplacian via finite differences
d = x[0,1] - x[0,0]
lap = (np.roll(f,1,0) + np.roll(f,-1,0) + np.roll(f,1,1) + np.roll(f,-1,1) - 4*f) / d**2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Laplacian of f = exp(−r²)', color='#f1f5f9', fontsize=13)

for ax, data, title, cmap in [
    (axes[0], f,   'f(x,y)',  'cool'),
    (axes[1], lap, 'Δf(x,y)', 'RdYlBu'),
    (axes[2], f + 0.05*lap, 'f + α·Δf\n(one diffusion step)', 'cool'),
]:
    im = ax.imshow(data, extent=[-3,3,-3,3], origin='lower', cmap=cmap)
    ax.set_title(title, color='#f1f5f9', fontsize=11)
    plt.colorbar(im, ax=ax)

plt.tight_layout(); plt.show()

# The Laplacian is negative at the peak (it's a maximum there)
cx = len(f)//2
print(f'f at origin:   {f[cx,cx]:.4f}')
print(f'Δf at origin:  {lap[cx,cx]:.4f}  (negative → local max ✓)')

In [ ]:
# ── Q60  Heat equation: ∂u/∂t = α·Δu ─────────────────────────────────────────
# Demonstrates how Laplacian drives diffusion / smoothing

N   = 100
dt  = 0.1
dx  = 1.0
alpha = 0.1
steps = [0, 5, 20, 100]

# Initial condition: two sharp peaks
u = np.zeros(N)
u[20] = 5.0
u[70] = 3.0

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Heat Equation  ∂u/∂t = α·Δu — Laplacian Drives Diffusion', 
             color='#f1f5f9', fontsize=13)

step_idx = 0
for t in range(max(steps)+1):
    if t in steps:
        ax = axes.flat[step_idx]
        ax.plot(u, color=BLUE, lw=2)
        ax.set_title(f't = {t} steps', color='#f1f5f9')
        ax.set_ylim(-0.5, 5.5)
        ax.set_xlabel('position')
        step_idx += 1
    # Explicit Euler for 1-D heat equation
    lap1d = np.roll(u,1) + np.roll(u,-1) - 2*u
    u = u + alpha * dt / dx**2 * lap1d

plt.tight_layout(); plt.show()
print('Peaks spread and decay over time — this is the Laplacian in action.')

---
## 8  Hessian  [Q66–Q70]

**Hessian H[i,j] = ∂²f/∂xᵢ∂xⱼ**

- Symmetric matrix (Schwarz/Clairaut theorem: mixed partials commute)  
- Encodes **curvature** of the function  
- trace(H) = **Laplacian**  
- Used in second-order optimization (Newton's method)

In [ ]:
# ── Q67  Hessian: bowl vs saddle ──────────────────────────────────────────────

x, y = np.meshgrid(np.linspace(-2, 2, 80), np.linspace(-2, 2, 80))

f_bowl   =  x**2 + y**2          # H = [[2,0],[0,2]]  pos-def  → minimum
f_saddle =  x**2 - y**2          # H = [[2,0],[0,-2]] indef    → saddle

fig = plt.figure(figsize=(14, 6))
fig.suptitle('Hessian Eigenvalues Determine Critical Point Type', 
             color='#f1f5f9', fontsize=13)

for idx, (Z, title) in enumerate([
    (f_bowl,   'f = x²+y²\nH pos-def → minimum'),
    (f_saddle, 'f = x²−y²\nH indefinite → saddle'),
]):
    ax = fig.add_subplot(1, 2, idx+1, projection='3d')
    ax.set_facecolor('#0f172a')
    ax.plot_surface(x, y, Z, cmap='cool', alpha=0.8, edgecolor='none')
    ax.set_title(title, color='#f1f5f9', pad=10)
    ax.tick_params(colors='#64748b')

plt.tight_layout(); plt.show()

# Compute Hessian analytically
print('Bowl f = x²+y²:')
H_bowl = np.array([[2, 0], [0, 2]])
eigvals, _ = np.linalg.eigh(H_bowl)
print(f'  H = {H_bowl.tolist()},  eigenvalues = {eigvals},  trace = {np.trace(H_bowl)} (= Laplacian)')

print('\nSaddle f = x²−y²:')
H_saddle = np.array([[2, 0], [0, -2]])
eigvals, _ = np.linalg.eigh(H_saddle)
print(f'  H = {H_saddle.tolist()},  eigenvalues = {eigvals},  trace = {np.trace(H_saddle)} (= Laplacian)')

In [ ]:
# ── Newton's method uses the Hessian ─────────────────────────────────────────
# x_{k+1} = x_k − H(x_k)^{-1} ∇f(x_k)

def f_opt(x):  return (x[0] - 1)**2 + 10*(x[1] - x[0]**2)**2   # Rosenbrock

def grad_f(x):
    dx0 = 2*(x[0]-1) - 40*x[0]*(x[1] - x[0]**2)
    dx1 = 20*(x[1] - x[0]**2)
    return np.array([dx0, dx1])

def hess_f(x):
    h00 = 2 - 40*(x[1] - x[0]**2) + 80*x[0]**2
    h01 = -40*x[0]
    h11 = 20.0
    return np.array([[h00, h01], [h01, h11]])

# Newton's method
xk = np.array([-1., 2.])
traj = [xk.copy()]
for _ in range(20):
    g = grad_f(xk)
    H = hess_f(xk)
    xk = xk - np.linalg.solve(H, g)
    traj.append(xk.copy())

traj = np.array(traj)

x0r, x1r = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-0.5, 2.5, 200))
Z = (x0r - 1)**2 + 10*(x1r - x0r**2)**2

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_title("Newton's Method on Rosenbrock — Hessian guides fast convergence", 
             color='#f1f5f9')
ax.contourf(x0r, x1r, np.log1p(Z), levels=30, cmap='inferno', alpha=0.7)
ax.contour( x0r, x1r, np.log1p(Z), levels=15, colors='#475569', linewidths=0.5)
ax.plot(traj[:,0], traj[:,1], 'o-', color=BLUE, lw=2, ms=5, label='Newton steps')
ax.scatter([1], [1], color=GREEN, s=120, zorder=5, label='Minimum (1,1)')
ax.legend(framealpha=0.2); ax.set_xlabel('x₀'); ax.set_ylabel('x₁')
plt.tight_layout(); plt.show()
print(f'Converged to: {xk}  (truth: [1,1])')
print(f'Steps taken:  {len(traj)-1}')

---
## Summary Table

| Operator | Definition | In CG |
|---|---|---|
| **Gradient** ∇f | Vector of partial derivatives | Steepest ascent, normal to surface |
| **Divergence** ∇·F | Sum of partials ∂Fᵢ/∂xᵢ | Sources/sinks in flow |
| **Curl** ∇×F | Anti-symmetric partials | Rotation in fluid/EM |
| **Laplacian** Δf | div(grad f) = Σ∂²f/∂xᵢ² | Heat/diffusion, smoothness |
| **Hessian** H | Matrix of 2nd partials | Curvature, Newton's method |

**Key identity:** `div(curl F) = 0` and `curl(grad f) = 0`